# # Configuração Inicial do Projeto F1 Databricks
# Este notebook configura o ambiente, catálogos e schemas necessários

In [0]:
CREATE CATALOG IF NOT EXISTS f1_lakehouse
COMMENT 'Catálogo criado para armazenar bases de dados do projeto';
USE CATALOG f1_lakehouse;

In [0]:
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Schema para dados bronze';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Schema para dados silver';
CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Schema para dados gold';

In [0]:
select * from bronze.ingestion_control

In [0]:
-- Tabela de controle de ingestão
CREATE TABLE IF NOT EXISTS bronze.ingestion_control (
    source_name STRING,
    season INT,
    endpoint STRING,
    records_ingested INT,
    ingested_at TIMESTAMP,
    status STRING,
    error_message STRING
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Bronze: drivers_raw
CREATE TABLE IF NOT EXISTS bronze.drivers_raw (
    driverId STRING,
    driverRef STRING,
    number STRING,
    code STRING,
    forename STRING,
    surname STRING,
    nationality STRING,
    dateOfBirth STRING,
    url STRING,
    _ingested_at TIMESTAMP,
    _source STRING,
    _season INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Bronze: constructors_raw
CREATE TABLE IF NOT EXISTS bronze.constructors_raw (
    constructorId STRING,
    constructorRef STRING,
    name STRING,
    nationality STRING,
    url STRING,
    _ingested_at TIMESTAMP,
    _source STRING,
    _season INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Bronze: races_raw
CREATE TABLE IF NOT EXISTS bronze.races_raw (
    raceId STRING,
    year STRING,
    round STRING,
    circuitId STRING,
    name STRING,
    date STRING,
    time STRING,
    url STRING,
    _ingested_at TIMESTAMP,
    _source STRING,
    _season INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Bronze: results_raw
CREATE TABLE IF NOT EXISTS bronze.results_raw (
    resultId STRING,
    raceId STRING,
    driverId STRING,
    constructorId STRING,
    number STRING,
    grid STRING,
    position STRING,
    positionText STRING,
    positionOrder STRING,
    points STRING,
    laps STRING,
    time STRING,
    milliseconds STRING,
    fastestLap STRING,
    rank STRING,
    fastestLapTime STRING,
    fastestLapSpeed STRING,
    statusId STRING,
    _ingested_at TIMESTAMP,
    _source STRING,
    _season INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Silver: drivers (será criada pela DLT)
CREATE TABLE IF NOT EXISTS silver.drivers (
    driver_id STRING,
    number STRING,
    code STRING,
    full_name STRING,
    forename STRING,
    surname STRING,
    nationality STRING,
    date_of_birth DATE,
    _ingested_at TIMESTAMP,
    _source STRING,
    _season INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Silver: constructors (será criada pela DLT)
CREATE TABLE IF NOT EXISTS silver.constructors (
    constructor_id STRING,
    name STRING,
    nationality STRING,
    _ingested_at TIMESTAMP,
    _source STRING,
    _season INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Silver: races (será criada pela DLT, com particionamento)
CREATE TABLE IF NOT EXISTS silver.races (
    race_id STRING,
    season INT,
    round STRING,
    circuit_id STRING,
    race_name STRING,
    race_date DATE,
    race_time STRING,
    race_timestamp TIMESTAMP,
    url STRING,
    _ingested_at TIMESTAMP
) USING DELTA
PARTITIONED BY (season)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Silver: results (será criada pela DLT, com particionamento)
CREATE TABLE IF NOT EXISTS silver.results (
    result_id STRING,
    race_id STRING,
    driver_id STRING,
    constructor_id STRING,
    car_number STRING,
    grid_position STRING,
    final_position INT,
    position_text STRING,
    position_order STRING,
    points FLOAT,
    laps STRING,
    race_time STRING,
    milliseconds STRING,
    fastest_lap STRING,
    fastest_lap_rank STRING,
    fastest_lap_time STRING,
    fastest_lap_speed STRING,
    status_id STRING,
    season INT,
    race_name STRING,
    race_date DATE,
    driver_name STRING,
    _ingested_at TIMESTAMP,
    _source STRING
) USING DELTA
PARTITIONED BY (season)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Gold: driver_standings
CREATE TABLE IF NOT EXISTS gold.driver_standings (
    _season INT,
    driver_id STRING,
    total_points FLOAT,
    races_participated INT,
    wins INT,
    podiums INT,
    avg_finish_position FLOAT,
    rank INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);

-- Gold: constructor_standings
CREATE TABLE IF NOT EXISTS gold.constructor_standings (
    _season INT,
    constructor_id STRING,
    total_points FLOAT,
    races_participated INT,
    wins INT,
    podiums INT,
    avg_finish_position FLOAT,
    rank INT
) USING DELTA
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true'
);